In [1]:
import sys
import pandas as pd
from google.colab import drive
from sklearn.preprocessing import StandardScaler

In [2]:
drive.mount('/content/drive')
views_path = '/content/drive/MyDrive/機器學習小組/views_predict.csv'
subs_path = '/content/drive/MyDrive/機器學習小組/subscribers_predict.csv'
excel_path = '/content/drive/MyDrive/機器學習小組/new_travel.xlsx'
cluster_path = '/content/drive/MyDrive/機器學習小組/creator_cluster.csv'

Mounted at /content/drive


In [3]:
# === 讀取資料 ===
df_views = pd.read_csv(views_path)
df_subs = pd.read_csv(subs_path)
df_excel = pd.read_excel(excel_path).dropna()
cluster_df = pd.read_csv(cluster_path)
display(df_views)
display(df_subs)
display(df_excel)
display(cluster_df)

,creator_handle,Actual_Views_2025_04,Predicted_Views_2025_04
0,tonyhuang38,989485.0,1.514645e+06
1,27apt,858483.0,1.995632e+05
2,campfiretw,795852.0,1.334756e+06
3,caitaitai945,662498.0,2.693739e+05
4,keatfilms9821,822440.0,1.357426e+06
...,...,...,...
64,yuniquecc,616752.0,1.355924e+06
65,0323matzu,258863.0,-1.844692e+05
66,theliupei,453514.0,-7.274228e+03
67,aikygo,420318.0,2.563434e+05


,creator_handle,Actual_Subscribers_2025_04,Predicted_Subscribers_2025_04
0,tonyhuang38,1000.0,3677.198516
1,27apt,4800.0,3088.158758
2,campfiretw,1000.0,2383.957498
3,caitaitai945,4000.0,1114.663694
4,keatfilms9821,1000.0,1029.213223
...,...,...,...
64,yuniquecc,3000.0,6198.897239
65,0323matzu,800.0,748.794776
66,theliupei,0.0,-347.785166
67,aikygo,1000.0,1848.873720


,creator_handle,videos,subscribers,views,2022-05-31_subscribers,2022-05-31_views,2022-05-31_videos,2022-06-30_subscribers,2022-06-30_views,2022-06-30_videos,...,2025-01-31_videos,2025-02-28_subscribers,2025-02-28_views,2025-02-28_videos,2025-03-31_subscribers,2025-03-31_views,2025-03-31_videos,2025-04-30_subscribers,2025-04-30_views,2025-04-30_videos
0,tonyhuang38,1589,238000,73434505,0.0,0.0,0.0,2000.0,622906.0,0.0,...,0,2000.0,996573.0,1517.0,4000,1386245,23,1000,989485,23
1,27apt,988,39300,11522637,0.0,0.0,0.0,540.0,152353.0,0.0,...,0,1100.0,294622.0,952.0,3600,390896,13,4800,858483,25
2,campfiretw,1247,308000,63030571,0.0,0.0,0.0,8000.0,1337563.0,0.0,...,0,2000.0,884778.0,1200.0,2000,1118578,18,1000,795852,14
3,caitaitai945,342,199000,44584362,0.0,0.0,0.0,3000.0,923106.0,0.0,...,0,1000.0,476800.0,325.0,1000,695216,6,4000,662498,4
4,keatfilms9821,299,258000,231688622,0.0,0.0,0.0,4000.0,5661685.0,0.0,...,0,1000.0,794918.0,298.0,1000,1290174,0,1000,822440,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,yuniquecc,292,136000,17651543,0.0,0.0,0.0,4500.0,438983.0,0.0,...,0,3000.0,442100.0,276.0,9000,980016,10,3000,616752,7
72,0323matzu,480,51800,17191840,0.0,0.0,0.0,200.0,64192.0,0.0,...,0,700.0,-117759.0,456.0,500,232925,6,800,258863,8
73,theliupei,1035,1220000,289195448,0.0,0.0,0.0,0.0,1433005.0,0.0,...,0,0.0,344711.0,1018.0,0,556965,7,0,453514,7
74,aikygo,794,277000,64069021,0.0,0.0,0.0,1000.0,490213.0,0.0,...,0,2000.0,586376.0,778.0,1000,577685,7,1000,420318,6


,creator_handle,videos,subscribers,views,cluster
0,tonyhuang38,1589,238000,73434505,0
1,27apt,988,39300,11522637,0
2,campfiretw,1247,308000,63030571,0
3,caitaitai945,342,199000,44584362,0
4,keatfilms9821,299,258000,231688622,0
...,...,...,...,...,...
64,yuniquecc,292,136000,17651543,0
65,0323matzu,480,51800,17191840,0
66,theliupei,1035,1220000,289195448,1
67,aikygo,794,277000,64069021,0


In [4]:
# === 擷取指定欄位，並依照 creator_handle 對齊順序 ===
df_base = df_excel[['creator_handle', '2025-03-31_videos']].copy()
df_base = df_base.set_index('creator_handle')
display(df_base)

# 整併所有預測資料到 df_base（保留 creator_handle 順序）
df_base['Predicted_Views_2025_04'] = df_views.set_index('creator_handle')['Predicted_Views_2025_04']
df_base['Predicted_Subscribers_2025_04'] = df_subs.set_index('creator_handle')['Predicted_Subscribers_2025_04']
display(df_base)

df_base = df_base.merge(cluster_df[['creator_handle', 'cluster']], on='creator_handle', how='left')
display(df_base)

# 移除 cluster = -1 的 outlier 創作者
df_base = df_base[df_base['cluster'] != -1]
display(df_base)
# 去除 cluster 欄位（可選）
df_base = df_base.drop(columns=['cluster'])
display(df_base)


,2025-03-31_videos
creator_handle,
tonyhuang38,23
27apt,13
campfiretw,18
caitaitai945,6
keatfilms9821,0
...,...
yuniquecc,10
0323matzu,6
theliupei,7


,2025-03-31_videos,Predicted_Views_2025_04,Predicted_Subscribers_2025_04
creator_handle,,,
tonyhuang38,23,1.514645e+06,3677.198516
27apt,13,1.995632e+05,3088.158758
campfiretw,18,1.334756e+06,2383.957498
caitaitai945,6,2.693739e+05,1114.663694
keatfilms9821,0,1.357426e+06,1029.213223
...,...,...,...
yuniquecc,10,1.355924e+06,6198.897239
0323matzu,6,-1.844692e+05,748.794776
theliupei,7,-7.274228e+03,-347.785166


,creator_handle,2025-03-31_videos,Predicted_Views_2025_04,Predicted_Subscribers_2025_04,cluster
0,tonyhuang38,23,1.514645e+06,3677.198516,0
1,27apt,13,1.995632e+05,3088.158758,0
2,campfiretw,18,1.334756e+06,2383.957498,0
3,caitaitai945,6,2.693739e+05,1114.663694,0
4,keatfilms9821,0,1.357426e+06,1029.213223,0
...,...,...,...,...,...
64,yuniquecc,10,1.355924e+06,6198.897239,0
65,0323matzu,6,-1.844692e+05,748.794776,0
66,theliupei,7,-7.274228e+03,-347.785166,1
67,aikygo,7,2.563434e+05,1848.873720,0


,creator_handle,2025-03-31_videos,Predicted_Views_2025_04,Predicted_Subscribers_2025_04,cluster
0,tonyhuang38,23,1.514645e+06,3677.198516,0
1,27apt,13,1.995632e+05,3088.158758,0
2,campfiretw,18,1.334756e+06,2383.957498,0
3,caitaitai945,6,2.693739e+05,1114.663694,0
4,keatfilms9821,0,1.357426e+06,1029.213223,0
...,...,...,...,...,...
64,yuniquecc,10,1.355924e+06,6198.897239,0
65,0323matzu,6,-1.844692e+05,748.794776,0
66,theliupei,7,-7.274228e+03,-347.785166,1
67,aikygo,7,2.563434e+05,1848.873720,0


,creator_handle,2025-03-31_videos,Predicted_Views_2025_04,Predicted_Subscribers_2025_04
0,tonyhuang38,23,1.514645e+06,3677.198516
1,27apt,13,1.995632e+05,3088.158758
2,campfiretw,18,1.334756e+06,2383.957498
3,caitaitai945,6,2.693739e+05,1114.663694
4,keatfilms9821,0,1.357426e+06,1029.213223
...,...,...,...,...
64,yuniquecc,10,1.355924e+06,6198.897239
65,0323matzu,6,-1.844692e+05,748.794776
66,theliupei,7,-7.274228e+03,-347.785166
67,aikygo,7,2.563434e+05,1848.873720


In [6]:
# ✅ 1. 指定數值欄位，不包含 creator_handle
numeric_cols = ['2025-03-31_videos', 'Predicted_Views_2025_04', 'Predicted_Subscribers_2025_04']

# ✅ 2. 保存創作者名稱（不是 index）
creator_names = df_base['creator_handle'].copy()  # ← 保留原始名稱欄位

# ✅ 3. 計算 Z 分數
scaler = StandardScaler()
z_scores = scaler.fit_transform(df_base[numeric_cols])

# ✅ 4. 組成新的 DataFrame 並加上 creator_handle
df_z = pd.DataFrame(
    z_scores,
    columns=[
        '2025-03-31_videos_z',
        'Predicted_Views_2025_04_z',
        'Predicted_Subscribers_2025_04_z'
    ]
)
df_z.index = creator_names  # 設定 index
df_z.index.name = 'creator_handle'
df_z = df_z.reset_index()
display(df_z)

,creator_handle,2025-03-31_videos_z,Predicted_Views_2025_04_z,Predicted_Subscribers_2025_04_z
0,tonyhuang38,0.095504,0.542447,0.955001
1,27apt,-0.085179,-0.599118,0.618972
2,campfiretw,0.005162,0.386293,0.217248
3,caitaitai945,-0.211657,-0.538519,-0.506844
4,keatfilms9821,-0.320067,0.405972,-0.555590
...,...,...,...,...
58,yuniquecc,-0.139384,0.404668,2.393549
59,0323matzu,-0.211657,-0.932480,-0.715560
60,theliupei,-0.193589,-0.778665,-1.341124
61,aikygo,-0.193589,-0.549830,-0.088000


In [7]:
# === 計算 decision index ===
df_z['decision_index'] = (
    0.6 * df_z['Predicted_Views_2025_04_z'] +
    0.3 * df_z['Predicted_Subscribers_2025_04_z'] +
    0.1 * df_z['2025-03-31_videos_z']
)
display(df_z)

,creator_handle,2025-03-31_videos_z,Predicted_Views_2025_04_z,Predicted_Subscribers_2025_04_z,decision_index
0,tonyhuang38,0.095504,0.542447,0.955001,0.621519
1,27apt,-0.085179,-0.599118,0.618972,-0.182297
2,campfiretw,0.005162,0.386293,0.217248,0.297466
3,caitaitai945,-0.211657,-0.538519,-0.506844,-0.496330
4,keatfilms9821,-0.320067,0.405972,-0.555590,0.044899
...,...,...,...,...,...
58,yuniquecc,-0.139384,0.404668,2.393549,0.946927
59,0323matzu,-0.211657,-0.932480,-0.715560,-0.795322
60,theliupei,-0.193589,-0.778665,-1.341124,-0.888895
61,aikygo,-0.193589,-0.549830,-0.088000,-0.375657


In [8]:
# === 整理輸出欄位順序並排序 ===
df_result = df_z.reset_index()[[
    'creator_handle',
    'Predicted_Views_2025_04_z',
    'Predicted_Subscribers_2025_04_z',
    '2025-03-31_videos_z',
    'decision_index'
]].sort_values(by='decision_index', ascending=False)

df_result = df_result.dropna()
display(df_result)

,creator_handle,Predicted_Views_2025_04_z,Predicted_Subscribers_2025_04_z,2025-03-31_videos_z,decision_index
27,ktstory,3.380787,2.355517,-0.157452,2.719382
21,nowyouon,2.477385,1.774097,-0.139384,2.004722
28,蔡淘貴,2.534124,1.023759,-0.211657,1.806436
20,gloryandy,1.799900,1.403918,-0.067111,1.494405
13,allieallie,1.671344,0.707619,-0.030974,1.211995
...,...,...,...,...,...
42,twobears,-0.841296,-1.222516,-0.229726,-0.894505
17,lostholic,-0.971908,-0.972585,-0.211657,-0.896086
9,6artstv,-1.323351,1.511827,-5.614084,-0.901871
36,hard2men1213,-1.242228,-0.662024,-0.103248,-0.954269


In [9]:
# === 儲存為 CSV 檔案到相同資料夾 ===
output_path = "/content/drive/MyDrive/機器學習小組/331_decision.csv"
df_result.to_csv(output_path, index=False)

display(df_result)

,creator_handle,Predicted_Views_2025_04_z,Predicted_Subscribers_2025_04_z,2025-03-31_videos_z,decision_index
27,ktstory,3.380787,2.355517,-0.157452,2.719382
21,nowyouon,2.477385,1.774097,-0.139384,2.004722
28,蔡淘貴,2.534124,1.023759,-0.211657,1.806436
20,gloryandy,1.799900,1.403918,-0.067111,1.494405
13,allieallie,1.671344,0.707619,-0.030974,1.211995
...,...,...,...,...,...
42,twobears,-0.841296,-1.222516,-0.229726,-0.894505
17,lostholic,-0.971908,-0.972585,-0.211657,-0.896086
9,6artstv,-1.323351,1.511827,-5.614084,-0.901871
36,hard2men1213,-1.242228,-0.662024,-0.103248,-0.954269
